# Gemma 4 SAE — Hub-first prompt feature dashboard

Enter a prompt, download the verified SAE release and its exact Gemma 4 revision from Hugging Face, capture the layer-20 residual stream, and inspect the resulting sparse features.

The visual workflow is inspired by Anthropic's feature dashboards in [Towards Monosemanticity](https://transformer-circuits.pub/2023/monosemantic-features/index.html): rank active features, inspect activation intensity across tokens, compare feature directions, and pair feature hypotheses with validation evidence. It is **not** an attribution graph. Anthropic's later [circuit-tracing work](https://www.anthropic.com/research/open-source-circuit-tracing) estimates causal edges using additional machinery; this notebook shows SAE activations and reconstruction quality at one hook point.

What this notebook downloads:

- the checksummed `lamm-mit` SAE release, labels, metrics, provenance, and example report;
- the exact gated `google/gemma-4-E4B` revision recorded by that release.

A live prompt therefore requires accepted Gemma access terms, a Hugging Face token, and enough memory for Gemma. CUDA is fastest; Apple MPS and CPU are supported but can be slow and memory-intensive.

## 1. Installation

From a clone of this repository, install the notebook dependencies and launch Jupyter:

```bash
python -m pip install -e ".[notebook]"
jupyter lab notebooks/hub_prompt_feature_dashboard.ipynb
```

In a hosted notebook without a clone, install directly from GitHub in a separate cell:

```python
%pip install "gemma4-sae[notebook] @ git+https://github.com/lamm-mit/gemma-sae.git"
```

Restart the kernel once after a first installation.

In [ ]:
import html
import json
from dataclasses import replace
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from huggingface_hub import get_token
from IPython.display import HTML, Markdown, clear_output, display

from gemma4_sae.config import load_config
from gemma4_sae.devices import select_device
from gemma4_sae.gemma import GemmaActivationExtractor, load_gemma
from gemma4_sae.label import label_lookup, load_label_registry
from gemma4_sae.release import load_release_bundle, resolve_release_bundle

# Hub source and output paths: keep them inside the notebook.
SAE_REPO_ID = "lamm-mit/gemma-4-e4b-layer20-batchtopk-sae"
SAE_REPO_REVISION = None  # Paste a Hub commit SHA for a frozen analysis.
HF_CACHE_DIR = None  # None uses the standard Hugging Face cache; or set a path here.
HF_LOCAL_FILES_ONLY = False  # True requires an already-cached release snapshot.
EXPORT_JSON_PATH = "prompt-feature-dashboard.json"

# Runtime. `auto` chooses CUDA, then Apple MPS, then CPU.
DEVICE = "auto"
DTYPE = "auto"
LOAD_IN_4BIT = False  # CUDA-only; requires the quantization extra.
MAX_TOKENS = 256
DEFAULT_TOP_FEATURES = 20

DEFAULT_PROMPT = (
    "A catalyst lowers the activation energy without changing the reaction "
    "equilibrium. Explain why this changes the reaction rate."
)

sns.set_theme(style="whitegrid", context="talk")
COLORS = {
    "ink": "#172554",
    "blue": "#2563eb",
    "cyan": "#0891b2",
    "gold": "#d97706",
    "red": "#dc2626",
    "green": "#059669",
    "muted": "#64748b",
}
plt.rcParams.update(
    {
        "figure.dpi": 115,
        "savefig.dpi": 300,
        "axes.titleweight": "bold",
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

print("Configuration ready. Edit the values above before continuing if needed.")

## 2. Authenticate and download the release

Accept the terms on the [Gemma 4 E4B model page](https://huggingface.co/google/gemma-4-E4B). If the next cell reports that no token is stored, run `from huggingface_hub import notebook_login; notebook_login()` in a new cell and enter a read token. Do not save a token inside this notebook.

`resolve_release_bundle` downloads the selected Hub revision and verifies every file against `checksums.json` before anything is analyzed.

In [ ]:
if get_token() is None:
    display(
        Markdown(
            "**No cached Hugging Face token was found.** Run `notebook_login()` "
            "in a new cell, then rerun this cell. The SAE repository is public, "
            "but the exact Gemma checkpoint is gated."
        )
    )
else:
    print("A cached Hugging Face token is available.")

analysis_device = select_device(DEVICE)
release_dir = resolve_release_bundle(
    SAE_REPO_ID,
    revision=SAE_REPO_REVISION,
    cache_dir=HF_CACHE_DIR,
    local_files_only=HF_LOCAL_FILES_ONLY,
)
config = load_config(release_dir / "resolved_config.json")
sae, activation_mean, activation_scale, sae_metadata = load_release_bundle(
    release_dir,
    device="cpu",
    verify=False,
)

labels_path = release_dir / "feature_labels.json"
if labels_path.is_file():
    label_registry = load_label_registry(labels_path)
    labels_by_id = label_lookup(label_registry)
else:
    label_registry = {"labels": []}
    labels_by_id = {}

def read_metrics(name):
    path = release_dir / name
    if not path.is_file():
        return {}
    report = json.loads(path.read_text())
    return report.get("metrics", report)


evaluation_metrics = read_metrics("evaluation.json")
fidelity_metrics = read_metrics("fidelity.json")
released_example_path = release_dir / "example_explanation.json"

release_summary = pd.DataFrame(
    [
        ("SAE repository", SAE_REPO_ID),
        ("SAE revision", SAE_REPO_REVISION or "main"),
        ("Base model", config.model.model_id),
        ("Base-model revision", config.model.revision),
        ("Hook layer", config.model.layer_index),
        ("Dictionary features", f"{sae.n_features:,}"),
        ("Training target L0", sae.target_l0),
        ("Released labels", f"{len(labels_by_id):,}"),
        ("Portable example", released_example_path.is_file()),
        ("Analysis device", str(analysis_device)),
    ],
    columns=["field", "value"],
)
display(release_summary.style.hide(axis="index"))
print("Verified release snapshot:", release_dir)

## 3. Load Gemma once

This is the expensive cell. It downloads the exact model revision recorded by the release and keeps the frozen model in memory so multiple prompts can be analyzed without reloading it. The SAE and Gemma are moved according to `DEVICE`; CUDA may use Accelerate's device map.

In [ ]:
model_config = replace(
    config.model,
    backend=DEVICE,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
runtime_config = replace(config, model=model_config)
runtime_config.validate()

tokenizer, model = load_gemma(model_config)
sae = sae.to(analysis_device).eval()
activation_mean = activation_mean.float().to(analysis_device)
activation_scale = activation_scale.float().reshape(()).to(analysis_device)
activation_scale = activation_scale.clamp_min(1e-8)

print("Loaded model:", model_config.model_id)
print("Model revision:", model_config.revision)
print("Hook:", f"decoder layer {model_config.layer_index} output")
print("SAE device:", next(sae.parameters()).device)
print("Inference threshold:", float(sae.inference_threshold))

In [ ]:
def decode_token(token_id):
    raw = tokenizer.convert_ids_to_tokens(int(token_id))
    try:
        text = tokenizer.decode(
            [int(token_id)],
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )
    except TypeError:
        text = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    return str(raw), str(text)


def label_fields(feature_id):
    record = labels_by_id.get(int(feature_id), {})
    interpretation = record.get("interpretation") or {}
    validation = record.get("validation") or {}
    return {
        "label": interpretation.get("label", "unlabeled"),
        "description": interpretation.get("description", ""),
        "activation_rule": interpretation.get("activation_rule", ""),
        "caveats": interpretation.get("caveats", []),
        "status": record.get("status", "unlabeled"),
        "balanced_accuracy": validation.get("balanced_accuracy"),
        "activation_prediction_spearman": validation.get(
            "activation_prediction_spearman"
        ),
        "validation": validation,
    }


@torch.inference_mode()
def analyze_prompt(text, top_n=20):
    text = text.strip()
    if not text:
        raise ValueError("Enter a non-empty prompt.")
    if top_n < 1:
        raise ValueError("top_n must be positive.")

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_TOKENS,
    )
    input_ids = encoded["input_ids"]
    attention_mask = encoded.get("attention_mask", torch.ones_like(input_ids))
    with GemmaActivationExtractor(model, model_config.layer_index) as extractor:
        hidden = extractor.extract(input_ids, attention_mask)

    normalized = (
        hidden.float().to(analysis_device) - activation_mean
    ) / activation_scale
    output = sae(normalized, use_threshold=True)

    normalized_cpu = normalized[0].float().cpu()
    reconstruction = output.reconstruction[0].float().cpu()
    features = output.features[0].float().cpu()
    residual = normalized_cpu - reconstruction
    token_ids = input_ids[0].tolist()
    special_ids = {int(token_id) for token_id in tokenizer.all_special_ids}

    token_rows = []
    for position, token_id in enumerate(token_ids):
        raw, decoded = decode_token(token_id)
        token_rows.append(
            {
                "position": position,
                "token_id": token_id,
                "token": raw,
                "text": decoded,
                "special": token_id in special_ids,
                "active_features": int((features[position] > 0).sum()),
                "reconstruction_cosine": float(
                    F.cosine_similarity(
                        normalized_cpu[position].unsqueeze(0),
                        reconstruction[position].unsqueeze(0),
                    )[0]
                ),
                "normalized_squared_error": float(
                    residual[position].square().sum()
                    / normalized_cpu[position].square().sum().clamp_min(1e-8)
                ),
            }
        )
    token_frame = pd.DataFrame(token_rows)

    maxima = features.max(dim=0).values
    masses = features.sum(dim=0)
    active_counts = (features > 0).sum(dim=0)
    active_ids = (maxima > 0).nonzero(as_tuple=False).flatten()
    count = min(int(top_n), int(active_ids.numel()))
    if count == 0:
        raise RuntimeError("No SAE features exceeded the released inference threshold.")
    ranked_ids = active_ids[
        torch.topk(maxima[active_ids], k=count, sorted=True).indices
    ]
    feature_ids = [int(feature_id) for feature_id in ranked_ids]
    total_mass = float(features.sum().clamp_min(1e-8))

    feature_rows = []
    for feature_id in feature_ids:
        values = features[:, feature_id]
        positive_positions = (values > 0).nonzero(as_tuple=False).flatten()
        context_count = min(5, int(positive_positions.numel()))
        if context_count:
            strongest = positive_positions[
                torch.topk(values[positive_positions], k=context_count).indices
            ].tolist()
        else:
            strongest = []
        fields = label_fields(feature_id)
        feature_rows.append(
            {
                "feature_id": feature_id,
                "label": fields["label"],
                "status": fields["status"],
                "max_activation": float(maxima[feature_id]),
                "activation_mass": float(masses[feature_id]),
                "mass_share": float(masses[feature_id]) / total_mass,
                "active_tokens": int(active_counts[feature_id]),
                "token_coverage": float(active_counts[feature_id]) / len(token_ids),
                "strongest_positions": strongest,
                "strongest_tokens": [token_rows[index]["text"] for index in strongest],
                "balanced_accuracy": fields["balanced_accuracy"],
                "activation_prediction_spearman": fields[
                    "activation_prediction_spearman"
                ],
                "description": fields["description"],
                "activation_rule": fields["activation_rule"],
                "caveats": fields["caveats"],
                "validation": fields["validation"],
            }
        )
    feature_frame = pd.DataFrame(feature_rows)
    activation_matrix = features[:, feature_ids].T.numpy()

    decoder_directions = sae.decoder.weight[:, feature_ids].detach().float().cpu().T
    decoder_directions = F.normalize(decoder_directions, dim=1)
    direction_similarity = (decoder_directions @ decoder_directions.T).numpy()

    squared_error = float(residual.square().sum())
    activation_energy = float(normalized_cpu.square().sum().clamp_min(1e-8))
    labeled_count = int((feature_frame["status"] != "unlabeled").sum())
    return {
        "prompt": text,
        "token_frame": token_frame,
        "feature_frame": feature_frame,
        "feature_ids": feature_ids,
        "activation_matrix": activation_matrix,
        "direction_similarity": direction_similarity,
        "features": features,
        "fve": 1.0 - squared_error / activation_energy,
        "normalized_mse": squared_error / activation_energy,
        "mean_cosine": float(token_frame["reconstruction_cosine"].mean()),
        "mean_l0": float(token_frame["active_features"].mean()),
        "labeled_count": labeled_count,
        "labeled_fraction": labeled_count / len(feature_ids),
        "inference_threshold": float(sae.inference_threshold),
    }


print("Prompt-analysis functions are ready.")

In [ ]:
def metric_card(label, value, note=""):
    return (
        "<div style='min-width:175px;padding:16px;border:1px solid #dbeafe;"
        "border-radius:12px;background:linear-gradient(135deg,#eff6ff,#fff)'>"
        f"<div style='font-size:12px;color:#64748b'>{html.escape(label)}</div>"
        f"<div style='font-size:26px;font-weight:750;color:#172554'>{value}</div>"
        f"<div style='font-size:12px;color:#64748b'>{html.escape(note)}</div></div>"
    )


def compact_feature_name(row):
    label = row["label"] if row["label"] != "unlabeled" else "unlabeled"
    if len(label) > 42:
        label = label[:39] + "…"
    return f"F{int(row['feature_id'])} · {label}"


def render_feature_inspector(result, feature_id):
    feature_id = int(feature_id)
    row_index = result["feature_ids"].index(feature_id)
    row = result["feature_frame"].iloc[row_index]
    activations = result["activation_matrix"][row_index]
    maximum = max(float(activations.max()), 1e-8)

    display(Markdown(f"### F{feature_id}: {row['label']}"))
    if row["description"]:
        display(Markdown(str(row["description"])))
    details = pd.DataFrame(
        [
            ("status", row["status"]),
            ("max activation", row["max_activation"]),
            ("activation mass", row["activation_mass"]),
            ("active tokens", row["active_tokens"]),
            ("balanced accuracy", row["balanced_accuracy"]),
            ("activation-prediction Spearman", row["activation_prediction_spearman"]),
        ],
        columns=["field", "value"],
    )
    display(details.style.hide(axis="index"))

    token_spans = []
    token_frame = result["token_frame"]
    for token, activation in zip(
        token_frame.to_dict("records"), activations, strict=True
    ):
        visible = token["text"].replace(" ", "␠").replace("\n", "↵")
        visible = visible or token["token"]
        intensity = float(activation) / maximum
        alpha = 0.04 if activation <= 0 else 0.12 + 0.78 * intensity
        token_spans.append(
            "<span title='position "
            f"{token['position']}; activation {float(activation):.4f}' "
            f"style='display:inline-block;margin:2px;padding:5px 7px;border-radius:5px;"
            f"background:rgba(220,38,38,{alpha:.3f});font-family:monospace'>"
            f"{html.escape(visible)}</span>"
        )
    display(HTML("<div style='line-height:2.2'>" + "".join(token_spans) + "</div>"))

    figure, axis = plt.subplots(
        figsize=(max(12, min(28, len(activations) * 0.45)), 4)
    )
    axis.plot(
        token_frame["position"],
        activations,
        color=COLORS["red"],
        marker="o",
        linewidth=2,
    )
    axis.fill_between(token_frame["position"], activations, color=COLORS["red"], alpha=0.15)
    axis.set_title(f"Activation trace for F{feature_id}")
    axis.set_xlabel("token position")
    axis.set_ylabel("SAE activation")
    plt.tight_layout()
    plt.show()

    if row["activation_rule"]:
        display(Markdown(f"**Proposed activation rule:** {row['activation_rule']}"))
    if row["caveats"]:
        display(Markdown("**Recorded caveats:** " + "; ".join(row["caveats"])))
    if not row["validation"]:
        display(
            Markdown(
                "*No released held-out validation score is available for this feature. "
                "Treat the label, if any, as a candidate hypothesis.*"
            )
        )


def render_dashboard(result):
    token_frame = result["token_frame"]
    feature_frame = result["feature_frame"]
    display(Markdown(f"## Prompt\n\n> {result['prompt']}"))
    heldout_l0 = evaluation_metrics.get("mean_l0")
    cards = (
        metric_card("Prompt tokens", str(len(token_frame)), "including special tokens")
        + metric_card("Mean prompt L0", f"{result['mean_l0']:.1f}", "threshold-active features")
        + metric_card("Reconstruction FVE", f"{result['fve']:.3f}", "this prompt at layer 20")
        + metric_card("Mean cosine", f"{result['mean_cosine']:.3f}", "activation vs reconstruction")
        + metric_card(
            "Labeled top features",
            f"{result['labeled_count']}/{len(result['feature_ids'])}",
            f"{result['labeled_fraction']:.0%} label coverage",
        )
    )
    display(HTML("<div style='display:flex;gap:12px;flex-wrap:wrap'>" + cards + "</div>"))
    if heldout_l0 is not None:
        display(
            Markdown(
                f"Released held-out mean L0: **{heldout_l0:.1f}**. Prompt L0 may differ "
                "because thresholded inference does not force a fixed top-k per token."
            )
        )

    figure, axes = plt.subplots(1, 2, figsize=(18, 5))
    colors = [COLORS["gold"] if value else COLORS["blue"] for value in token_frame["special"]]
    axes[0].bar(token_frame["position"], token_frame["active_features"], color=colors)
    if heldout_l0 is not None:
        axes[0].axhline(
            heldout_l0,
            color=COLORS["red"],
            linestyle="--",
            label=f"held-out mean {heldout_l0:.1f}",
        )
        axes[0].legend()
    axes[0].set_title("Sparse features active per token")
    axes[0].set_xlabel("token position")
    axes[0].set_ylabel("L0")

    axes[1].plot(
        token_frame["position"],
        token_frame["reconstruction_cosine"],
        color=COLORS["green"],
        marker="o",
        label="cosine similarity",
    )
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].set_title("Per-token reconstruction fidelity")
    axes[1].set_xlabel("token position")
    axes[1].set_ylabel("cosine similarity")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    display(Markdown("## Strongest prompt features"))
    visible_columns = [
        "feature_id",
        "label",
        "status",
        "max_activation",
        "activation_mass",
        "active_tokens",
        "strongest_positions",
        "balanced_accuracy",
        "activation_prediction_spearman",
    ]
    display(
        feature_frame[visible_columns]
        .style.format(
            {
                "max_activation": "{:.3f}",
                "activation_mass": "{:.3f}",
                "balanced_accuracy": lambda value: "—" if pd.isna(value) else f"{value:.2f}",
                "activation_prediction_spearman": lambda value: (
                    "—" if pd.isna(value) else f"{value:.2f}"
                ),
            }
        )
        .background_gradient(subset=["max_activation"], cmap="Blues")
        .hide(axis="index")
    )

    matrix = result["activation_matrix"]
    row_maxima = np.maximum(matrix.max(axis=1, keepdims=True), 1e-8)
    relative_matrix = matrix / row_maxima
    ylabels = [compact_feature_name(row) for _, row in feature_frame.iterrows()]
    xlabels = [
        f"{row.position}:{(row.text or row.token).replace(chr(10), '↵')}"
        for row in token_frame.itertuples()
    ]
    plt.figure(
        figsize=(
            max(14, min(30, len(token_frame) * 0.6)),
            max(7, len(feature_frame) * 0.42),
        )
    )
    sns.heatmap(
        relative_matrix,
        xticklabels=xlabels,
        yticklabels=ylabels,
        cmap="mako",
        vmin=0,
        vmax=1,
        linewidths=0.2,
        cbar_kws={"label": "activation / feature's prompt maximum"},
    )
    plt.title("Anthropic-style view: where each top feature activates")
    plt.xlabel("token position and decoded token")
    plt.ylabel("SAE feature")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(max(9, len(feature_frame) * 0.45), max(8, len(feature_frame) * 0.42)))
    sns.heatmap(
        result["direction_similarity"],
        xticklabels=[f"F{feature_id}" for feature_id in result["feature_ids"]],
        yticklabels=[f"F{feature_id}" for feature_id in result["feature_ids"]],
        cmap="vlag",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        cbar_kws={"label": "decoder-direction cosine similarity"},
    )
    plt.title("Geometry of the selected SAE decoder directions")
    plt.tight_layout()
    plt.show()
    display(
        Markdown(
            "Direction similarity can reveal redundant or related dictionary vectors, "
            "but it is not a causal edge or evidence that one feature computes another."
        )
    )

    options = [
        (compact_feature_name(row), int(row["feature_id"]))
        for _, row in feature_frame.iterrows()
    ]
    feature_selector = widgets.Dropdown(
        options=options,
        value=options[0][1],
        description="Inspect:",
        layout=widgets.Layout(width="95%"),
        style={"description_width": "80px"},
    )
    feature_output = widgets.Output()

    def update_feature(change=None):
        del change
        with feature_output:
            clear_output(wait=True)
            render_feature_inspector(result, feature_selector.value)

    feature_selector.observe(update_feature, names="value")
    display(Markdown("## Inspect one feature"))
    display(feature_selector, feature_output)
    update_feature()


print("Visualization functions are ready.")

## 4. Enter a prompt and analyze it

Edit the prompt in the text box and click **Analyze prompt**. Gemma stays loaded, so subsequent prompts avoid the model-loading cost. The dashboard ranks features by maximum activation on this prompt. Ranking is a discovery aid—not a statistical significance test.

In [ ]:
prompt_box = widgets.Textarea(
    value=DEFAULT_PROMPT,
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="150px"),
    style={"description_width": "70px"},
)
top_features_slider = widgets.IntSlider(
    value=DEFAULT_TOP_FEATURES,
    min=5,
    max=40,
    step=1,
    description="Top features:",
    continuous_update=False,
    style={"description_width": "100px"},
)
run_button = widgets.Button(
    description="Analyze prompt",
    button_style="primary",
    icon="microscope",
)
dashboard_output = widgets.Output()
DASHBOARD_STATE = {}

def run_dashboard(_button=None):
    with dashboard_output:
        clear_output(wait=True)
        print("Running Gemma and the SAE…")
        try:
            result = analyze_prompt(
                prompt_box.value,
                top_n=top_features_slider.value,
            )
        except Exception as error:
            display(
                HTML(
                    "<div style='padding:14px;border-left:5px solid #dc2626;"
                    f"background:#fef2f2'><b>Analysis failed:</b> {html.escape(str(error))}</div>"
                )
            )
            raise
        clear_output(wait=True)
        DASHBOARD_STATE["result"] = result
        render_dashboard(result)


run_button.on_click(run_dashboard)
display(widgets.VBox([prompt_box, top_features_slider, run_button, dashboard_output]))
run_dashboard()

## 5. Export a compact analysis record

The export contains the prompt, release identity, aggregate reconstruction metrics, token diagnostics, and ranked feature table. It intentionally omits the full 30,720-dimensional activation tensor. Review prompt sensitivity before sharing the JSON.

In [ ]:
def export_dashboard(path=EXPORT_JSON_PATH):
    if "result" not in DASHBOARD_STATE:
        raise RuntimeError("Analyze a prompt before exporting.")
    result = DASHBOARD_STATE["result"]
    payload = {
        "format_version": 1,
        "sae_repo_id": SAE_REPO_ID,
        "sae_revision": SAE_REPO_REVISION or "main",
        "model_id": config.model.model_id,
        "model_revision": config.model.revision,
        "layer_index": config.model.layer_index,
        "prompt": result["prompt"],
        "metrics": {
            "mean_l0": result["mean_l0"],
            "fve": result["fve"],
            "normalized_mse": result["normalized_mse"],
            "mean_cosine": result["mean_cosine"],
            "inference_threshold": result["inference_threshold"],
            "labeled_fraction": result["labeled_fraction"],
        },
        "tokens": result["token_frame"].to_dict("records"),
        "features": result["feature_frame"].to_dict("records"),
    }
    destination = Path(path)
    destination.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str))
    print("Wrote:", destination.resolve())
    return destination


# Uncomment to save the current dashboard result.
# export_dashboard()

## How to turn a dashboard observation into a scientific finding

Use this notebook for **hypothesis generation**:

1. Identify a strongly and selectively activated feature on the target tokens.
2. Read its released label status and held-out scores; `candidate` and `unlabeled` are not validated interpretations.
3. Change one semantic property at a time and rerun matched control prompts. Check whether the feature follows the hypothesized property rather than token identity, position, punctuation, or formatting.
4. Mine independent positive examples, hard negatives, and zero-activation controls on the training machine. Freeze these contexts before final evaluation.
5. Reproduce across documents, datasets, and SAE seeds. Inspect feature splitting and nearby decoder directions.
6. For causal claims, intervene on the feature at the trained hook and compare matched/random-direction controls. Activation alone establishes correlation, not causality.

### Method boundary

Anthropic's 2023 feature visualizations combined activating examples, activation intervals, automated explanations, held-out activation prediction, downstream effects, and cross-run relationships. This notebook implements the parts supported by the released Gemma 4 residual-stream SAE: full prompt activations, reconstruction diagnostics, checkpoint-bound labels and validation scores, and decoder-direction geometry. The release intentionally excludes raw mined contexts for privacy and licensing reasons.

Anthropic's 2025 attribution graphs go further by estimating feature-to-feature computational edges and testing interventions. A token-feature heatmap is not such a graph and should not be described as one.

Primary references:

- [Towards Monosemanticity: Decomposing Language Models With Dictionary Learning](https://transformer-circuits.pub/2023/monosemantic-features/index.html)
- [Dictionary Learning Features visualization](https://transformer-circuits.pub/2023/monosemantic-features/vis/)
- [Open-sourcing circuit-tracing tools](https://www.anthropic.com/research/open-source-circuit-tracing)
- [Scaling and evaluating sparse autoencoders](https://arxiv.org/abs/2406.04093)
